In [34]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
from collections import Counter
import os
import ast

In [35]:
import importlib
import new_solution_scoring
import mrs

importlib.reload(new_solution_scoring)
importlib.reload(mrs)

from new_solution_scoring import (
    load_front_and_solution,
    compute_matchup_outcomes,
    compute_coalition_coverage,
)
from mrs import (
    density_filter,
    _safe_iqr
)

In [36]:
num_subjects = 10
num_trials = 17

converters = {
    "recommended_values": ast.literal_eval,
    "objective_names": ast.literal_eval,
}
sub_data_all = pd.read_csv('data/sub_data_all.csv', converters=converters)

subject_mapping = (
    sub_data_all[["sub_id", "original_sub_id"]]
    .drop_duplicates()
    .sort_values("sub_id")
)

In [ ]:
for i, row in enumerate(subject_mapping.itertuples(index=False)):
    for trial_id in range(1, num_trials + 1):
        sub_id = int(row.sub_id)
        original_sub_id = int(row.original_sub_id)
        print("sub:", sub_id, "trial:",trial_id)

        new_solution = sub_data_all.loc[
            (sub_data_all["sub_id"] == sub_id) &
            (sub_data_all["trial_id"] == trial_id)
        ]["recommended_values"].iloc[-1]
        objectives = sub_data_all["objective_names"].iloc[0]

        pareto_path = f"data/game_data/pf_block_2_trial_{trial_id}_*.csv"

        X_full, new, objectives = load_front_and_solution(
            pareto_path, objectives, new_solution
        )
        X, _ = density_filter(X_full, 0.8)

        N, D = X.shape
        new_wins, front_win, draws, gt, wins_j, m, win_pct = compute_matchup_outcomes(X, new)
        coalitions, coverage, total_wins = compute_coalition_coverage(gt, new_wins, objectives, m)

        
        

sub: 0 trial: 1
sub: 0 trial: 2
sub: 0 trial: 3
[{'indices': (0, 1, 2), 'label': 'REB+AST+STL', 'bitmask': 7}, {'indices': (0, 1, 3), 'label': 'REB+AST+BLK', 'bitmask': 11}, {'indices': (0, 1, 4), 'label': 'REB+AST+PTS', 'bitmask': 19}, {'indices': (0, 2, 3), 'label': 'REB+STL+BLK', 'bitmask': 13}, {'indices': (0, 2, 4), 'label': 'REB+STL+PTS', 'bitmask': 21}, {'indices': (0, 3, 4), 'label': 'REB+BLK+PTS', 'bitmask': 25}, {'indices': (1, 2, 3), 'label': 'AST+STL+BLK', 'bitmask': 14}, {'indices': (1, 2, 4), 'label': 'AST+STL+PTS', 'bitmask': 22}, {'indices': (1, 3, 4), 'label': 'AST+BLK+PTS', 'bitmask': 26}, {'indices': (2, 3, 4), 'label': 'STL+BLK+PTS', 'bitmask': 28}]
Counter({13: 5729, 7: 515, 14: 293, 11: 227})
0.8233468286099865
sub: 0 trial: 4
sub: 0 trial: 5
sub: 0 trial: 6
sub: 0 trial: 7
sub: 0 trial: 8
sub: 0 trial: 9
sub: 0 trial: 10
sub: 0 trial: 11
sub: 0 trial: 12
sub: 0 trial: 13
sub: 0 trial: 14
sub: 0 trial: 15
sub: 0 trial: 16
sub: 0 trial: 17
sub: 1 trial: 1
sub: 1 tr

In [37]:
records = []

for row in subject_mapping.itertuples(index=False):
    for trial_id in range(1, num_trials + 1):
        sub_id = int(row.sub_id)
        print("sub:", sub_id, "trial:", trial_id)

        new_solution = sub_data_all.loc[
            (sub_data_all["sub_id"] == sub_id) &
            (sub_data_all["trial_id"] == trial_id)
        ]["recommended_values"].iloc[-1]
        objectives = sub_data_all["objective_names"].iloc[0]

        pareto_path = f"data/game_data/pf_block_2_trial_{trial_id}_*.csv"
        X_full, new, objectives = load_front_and_solution(
            pareto_path, objectives, new_solution
        )
        X, _ = density_filter(X_full, 0.8)

        new_wins, front_win, draws, gt, wins_j, m, win_pct = compute_matchup_outcomes(X, new)
        coalitions, coverage, total_wins = compute_coalition_coverage(gt, new_wins, objectives, m)

        record = {
            "sub_id": sub_id,
            "trial_id": trial_id,
            "win_pct": win_pct,
        }

        if total_wins > 0:
            winning = max(coalitions, key=lambda c: coverage[c["bitmask"]])
            record["winning_coalition_label"] = winning["label"]
            record["winning_coalition_indices"] = winning["indices"]
        else:
            record["winning_coalition_label"] = None
            record["winning_coalition_indices"] = None

        for c in coalitions:
            col = f"coalition_coverage_pct_{c['label']}"
            record[col] = coverage[c["bitmask"]] / total_wins if total_wins else 0.0

        records.append(record)

id_cols = ["sub_id", "trial_id", "winning_coalition_label", "winning_coalition_indices", "win_pct"]
results_df = pd.DataFrame(records)
coverage_cols = [c for c in results_df.columns if c.startswith("coalition_coverage_pct_")]
results_df = results_df[id_cols + coverage_cols]
results_df


sub: 0 trial: 1
sub: 0 trial: 2
sub: 0 trial: 3
sub: 0 trial: 4
sub: 0 trial: 5
sub: 0 trial: 6
sub: 0 trial: 7
sub: 0 trial: 8
sub: 0 trial: 9
sub: 0 trial: 10
sub: 0 trial: 11
sub: 0 trial: 12
sub: 0 trial: 13
sub: 0 trial: 14
sub: 0 trial: 15
sub: 0 trial: 16
sub: 0 trial: 17
sub: 1 trial: 1
sub: 1 trial: 2
sub: 1 trial: 3
sub: 1 trial: 4
sub: 1 trial: 5
sub: 1 trial: 6
sub: 1 trial: 7
sub: 1 trial: 8
sub: 1 trial: 9
sub: 1 trial: 10
sub: 1 trial: 11
sub: 1 trial: 12
sub: 1 trial: 13
sub: 1 trial: 14
sub: 1 trial: 15
sub: 1 trial: 16
sub: 1 trial: 17
sub: 2 trial: 1
sub: 2 trial: 2
sub: 2 trial: 3
sub: 2 trial: 4
sub: 2 trial: 5
sub: 2 trial: 6
sub: 2 trial: 7
sub: 2 trial: 8
sub: 2 trial: 9
sub: 2 trial: 10
sub: 2 trial: 11
sub: 2 trial: 12
sub: 2 trial: 13
sub: 2 trial: 14
sub: 2 trial: 15
sub: 2 trial: 16
sub: 2 trial: 17
sub: 3 trial: 1
sub: 3 trial: 2
sub: 3 trial: 3
sub: 3 trial: 4
sub: 3 trial: 5
sub: 3 trial: 6
sub: 3 trial: 7
sub: 3 trial: 8
sub: 3 trial: 9
sub: 3 trial: 10

,sub_id,trial_id,winning_coalition_label,winning_coalition_indices,win_pct,coalition_coverage_pct_REB+AST+STL,coalition_coverage_pct_REB+AST+BLK,coalition_coverage_pct_REB+AST+PTS,coalition_coverage_pct_REB+STL+BLK,coalition_coverage_pct_REB+STL+PTS,coalition_coverage_pct_REB+BLK+PTS,coalition_coverage_pct_AST+STL+BLK,coalition_coverage_pct_AST+STL+PTS,coalition_coverage_pct_AST+BLK+PTS,coalition_coverage_pct_STL+BLK+PTS
0,0,1,AST+STL+PTS,"(1, 2, 4)",0.801813,0.001533,0.001341,0.000192,0.046944,0.027017,0.026825,0.180111,0.726384,0.107492,0.281088
1,0,2,AST+STL+PTS,"(1, 2, 4)",0.805764,0.000764,0.000509,0.001655,0.007255,0.015782,0.018328,0.028001,0.929871,0.029528,0.069874
2,0,3,REB+STL+BLK,"(0, 2, 3)",0.823347,0.084412,0.037207,0.000000,0.939026,0.000000,0.000000,0.048025,0.000000,0.000000,0.000000
3,0,4,AST+STL+PTS,"(1, 2, 4)",0.819430,0.000000,0.000000,0.000000,0.028414,0.034546,0.024734,0.044358,0.857931,0.039452,0.163123
4,0,5,REB+BLK+PTS,"(0, 3, 4)",0.855561,0.001798,0.000719,0.000719,0.421971,0.395361,0.642215,0.010428,0.052859,0.008091,0.414599
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,9,13,REB+STL+BLK,"(0, 2, 3)",0.775435,0.093003,0.068536,0.071398,0.357705,0.237945,0.333524,0.151095,0.189441,0.071541,0.215195
166,9,14,REB+STL+PTS,"(0, 2, 4)",0.912884,0.163289,0.034166,0.152686,0.301838,0.447691,0.446984,0.035815,0.251885,0.036287,0.211593
167,9,15,AST+STL+PTS,"(1, 2, 4)",0.747866,0.002506,0.005846,0.009187,0.031459,0.037305,0.318486,0.110802,0.441537,0.263085,0.205735
168,9,16,STL+BLK+PTS,"(2, 3, 4)",0.767246,0.003487,0.001245,0.003238,0.247073,0.290411,0.369116,0.049315,0.303861,0.057036,0.563636


In [41]:
coverage_cols = [c for c in results_df.columns if c.startswith("coalition_coverage_pct_")]
prefix = "coalition_coverage_pct_"

label_to_indices = (
    results_df.dropna(subset=["winning_coalition_label"])
    .drop_duplicates("winning_coalition_label")
    .set_index("winning_coalition_label")["winning_coalition_indices"]
    .to_dict()
)

grouped = results_df.groupby("sub_id", sort=True)
mean_win_pct = grouped["win_pct"].mean()
mean_coverage = grouped[coverage_cols].mean()
sum_coverage = grouped[coverage_cols].sum()

winning_col = sum_coverage.idxmax(axis=1)
winning_label = winning_col.str[len(prefix):]

mean_results_df = pd.DataFrame({
    "sub_id": mean_win_pct.index,
    "mean_win_pct": mean_win_pct.values,
    "mean_winning_coalition_label": winning_label.values,
    "mean_winning_coalition_indices": winning_label.map(label_to_indices).values,
})
for c in coverage_cols:
    label = c[len(prefix):]
    mean_results_df[f"mean_coalition_coverage_pct_{label}"] = mean_coverage[c].values

mean_results_df


,sub_id,mean_win_pct,mean_winning_coalition_label,mean_winning_coalition_indices,mean_coalition_coverage_pct_REB+AST+STL,mean_coalition_coverage_pct_REB+AST+BLK,mean_coalition_coverage_pct_REB+AST+PTS,mean_coalition_coverage_pct_REB+STL+BLK,mean_coalition_coverage_pct_REB+STL+PTS,mean_coalition_coverage_pct_REB+BLK+PTS,mean_coalition_coverage_pct_AST+STL+BLK,mean_coalition_coverage_pct_AST+STL+PTS,mean_coalition_coverage_pct_AST+BLK+PTS,mean_coalition_coverage_pct_STL+BLK+PTS
0,0,0.764981,AST+STL+PTS,"(1, 2, 4)",0.015841,0.006201,0.029695,0.168428,0.205086,0.322520,0.053673,0.350639,0.055374,0.214978
1,1,0.866876,AST+STL+PTS,"(1, 2, 4)",0.000026,0.001986,0.062187,0.000000,0.000230,0.046873,0.000000,0.882149,0.012536,0.000000
2,2,0.724805,AST+STL+PTS,"(1, 2, 4)",0.044463,0.166985,0.045628,0.163035,0.123292,0.262059,0.127321,0.289181,0.128216,0.183894
3,3,0.819934,REB+BLK+PTS,"(0, 3, 4)",0.018029,0.013531,0.022967,0.270534,0.291743,0.416716,0.056191,0.262332,0.060973,0.358699
4,4,0.607867,AST+STL+PTS,"(1, 2, 4)",0.012336,0.006979,0.018849,0.114184,0.181575,0.238579,0.077966,0.464910,0.039689,0.250750
5,5,0.702919,AST+STL+PTS,"(1, 2, 4)",0.019905,0.012285,0.020746,0.184016,0.249538,0.329251,0.055184,0.340866,0.068200,0.281563
6,6,0.802154,AST+STL+PTS,"(1, 2, 4)",0.000130,0.000044,0.000201,0.011441,0.093075,0.387837,0.001634,0.512980,0.001774,0.024889
7,7,0.796375,REB+BLK+PTS,"(0, 3, 4)",0.013743,0.015589,0.021948,0.223422,0.254065,0.374724,0.075107,0.278305,0.083001,0.322061
8,8,0.785790,AST+STL+PTS,"(1, 2, 4)",0.017835,0.020267,0.064857,0.191615,0.056406,0.102655,0.052602,0.589879,0.044589,0.091887
9,9,0.547032,REB+BLK+PTS,"(0, 3, 4)",0.072730,0.042557,0.069656,0.177988,0.242936,0.293788,0.066502,0.216303,0.054839,0.177032


In [ ]:
PAIRED_10 = sns.color_palette("Paired", n_colors=10)
PAIRED_10_HEX = [
    "#{:02x}{:02x}{:02x}".format(int(r * 255), int(g * 255), int(b * 255))
    for r, g, b in PAIRED_10
]

prefix = "mean_coalition_coverage_pct_"
coverage_cols = [c for c in mean_results_df.columns if c.startswith(prefix)]
coalition_labels = [c[len(prefix):] for c in coverage_cols]
y = np.arange(len(coalition_labels), dtype=float)
subject_colors = PAIRED_10[: len(mean_results_df)]

fig, ax = plt.subplots(figsize=(6, 5))
for i, (_, row) in enumerate(mean_results_df.iterrows()):
    values = row[coverage_cols].to_numpy(dtype=float)
    ax.plot(values, y, color=subject_colors[i], linewidth=1.5, zorder=2)
    ax.scatter(
        values, y,
        color=subject_colors[i], s=40, zorder=3,
        edgecolors="white", linewidths=0.4,
        label=f"Subject {int(row.sub_id) + 1}",
    )

ax.set_yticks(y)
ax.set_yticklabels(coalition_labels)
ax.set_xlabel("Mean coalition coverage pct over trials")
ax.set_ylabel("Coalition")
ax.set_xlim(-0.02, 1)
ax.invert_yaxis()
ax.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.show()
